# Tissue-coverage-filtered, bit-aligned WAI, and comparison to MERlin barcode counts

Follow-up to both `01_test_weighted_average_intensity.ipynb` (the original,
naive per-FOV WAI: a plain average of the 16 barcode bits' raw mean
intensity) and `04_test_intensity_vs_bit.ipynb` (found each bit's own
per-FOV intensity profile has roughly the same *shape* across FOVs, and can
be brought into close overlap with a single per-bit least-squares affine
rescale). This notebook combines both follow-ups into one refined WAI:

1. **Select FOVs**: drop partially-covered FOVs (mostly boundary/no-tissue)
   before computing anything else, using `MERci.analysis.fov.
   compute_tissue_fraction`'s true-pixel-count method (same one
   `misc/measure_tissue_thickness_test.ipynb`/`08_measure_tissue_thickness.
   ipynb` use for z-depth) to get a per-FOV `tissue_fraction` (0-1), then a
   *second*, separate threshold on that per-FOV distribution itself (not a
   pixel-intensity threshold) separating "fully covered" from "partially
   covered" FOVs -- estimated as the valley between the two most prominent
   modes of the `tissue_fraction` histogram (same valley-finding idea as
   `02_create_boundary_from_mosaic.ipynb`'s pixel-intensity threshold
   estimate, adapted here to linear-space fraction values rather than log10
   pixel intensity), falling back to Otsu's method if the distribution
   isn't clearly bimodal.
2. **Align each bit's profile** (only across the selected FOVs): reruns
   `04_test_intensity_vs_bit.ipynb` sections 6-7's exact least-squares
   affine overlay (min-subtract each bit's profile, rescale it onto the
   16-bit consensus shape with a single no-intercept least-squares scale
   factor) -- shows the same before/after plots as that notebook's own
   plots 6/7, plus a third plot: the aligned profiles again (thin grey,
   alpha=0.5) with their per-FOV mean overlaid as one bold curve -- the
   **weighted average intensity profile**. Its value at each FOV is this
   notebook's refined per-FOV WAI (each of the 16 bits already rescaled to
   a common, comparable scale before averaging, unlike the original WAI's
   plain average of raw bit intensities).
3. **Spatial heatmap** of that per-FOV WAI (same layout/style as
   `01_test_weighted_average_intensity.ipynb` section 6).
4. **Histogram** of the same WAI, for both datasets, checking (as the
   original WAI investigation set out to) whether its distribution shape
   flags a genuinely uneven-density region (bimodal/wide) vs. a uniform one
   (unimodal, low std).
5. **`BC555_sample_05/epi` only, for now**: compares this WAI to each FOV's
   total barcode count from MERlin's own `ExportBarcodes/barcodes.csv`
   (`BC553_sample_02/epi` has no barcode-count comparison here since step 5
   was scoped to BC555 only) -- assumes MERlin's own `fov` numbering matches
   MERci's `fov_id` (both derive from the same acquisition positions file/
   movie naming, the standard convention this pipeline's MERlin configs use).

**Datasets**: `BC555_sample_05/epi` and `BC553_sample_02/epi` -- neither
step needs MERlin `Decode`/`Optimize` output except step 5's barcode counts,
so `BC553_sample_02/epi`'s intact raw `data/` is enough for steps 1-4 even
though it's a separate re-processing run from `BC555_sample_05/epi`.

**Images produced, per dataset**: tissue-fraction distribution with its
selection threshold marked; per-bit intensity vs. FOV index before
alignment; the same after alignment; the aligned profiles with their
average overlaid; a spatial heatmap of the resulting WAI; a histogram of
the same. Plus, `BC555_sample_05/epi` only: WAI vs. MERlin barcode count
scatter with Pearson r.

Follows `NOTEBOOK_GUIDELINES.md` throughout: calculation cells are cached
under `analysis/cache/test_aligned_wai/` (tissue-fraction Counters instead
share `01`/`02`'s own `test_weighted_average_intensity` cache namespace,
see `TISSUE_CACHE_NAME` below) and skip already-done FOVs on rerun, with
`ProgressReporter` progress; display cells are separate and save every
figure to `analysis/figures/`.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.signal import find_peaks
from skimage.filters import threshold_otsu

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR = Path(os.getcwd()).parent.parent.parent   # MERci/

sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.io              import read_image_frames
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.analysis.fov           import compute_tissue_fraction, resolve_barcode_bit_lookup
from MERci.analysis.stage_z       import positions_to_grid_indices
from MERci.progress_display       import ProgressReporter
from MERci.visualization          import get_merci_figures_dir

NOTEBOOK_NAME = "test_aligned_wai"
print(f"MERCI_DIR : {MERCI_DIR}")

## 2 — Parameters

In [ ]:
# This is a test/investigation notebook (NOTEBOOK_GUIDELINES.md's "validation
# notebooks for a new feature" case under notebooks/tests/) -- it analyzes
# EXTERNAL datasets, not the experiment this MERci clone is deployed into, so
# SAMPLE_DIR is set explicitly per dataset rather than auto-detected from parent dirs.
BC555_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC555_sample_05/epi")
BC553_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC553_sample_02/epi")
IMAGE_SUFFIX = ".zarr"

# The 16 combinatorial MERFISH bits (codebook_0_C3v1_codebook.csv has exactly
# 16 RS-readout columns); round_bit_color_map.csv's bit numbers 17+ are the
# non-combinatorial "sequential genes" readout, excluded here.
N_BARCODE_BITS = 16

Z_PLANE = None   # None = auto-pick the middle z of each dataset's own bits-round frame table

# Tissue-fraction pixel threshold (MERci.analysis.fov.compute_tissue_fraction)
# -- same default as misc/measure_tissue_thickness_test.ipynb / 08_measure_tissue_thickness.ipynb.
N_BACKGROUND_FRAMES = 10
COVERAGE_HIST_BINS  = 40   # bins for the per-FOV tissue_fraction distribution (step 1)

# Reuse 01_test_weighted_average_intensity.ipynb's own tissue-fraction cache
# namespace instead of a fresh one under this notebook's name, so whichever
# notebook computes a given FOV's DAPI Counter first, the others reuse it.
TISSUE_CACHE_NAME = "test_weighted_average_intensity"

# BC555_sample_05/epi only (step 5) -- MERlin's own exported per-barcode table.
MERLIN_BARCODES_PATH_555 = BC555_DIR / "merlin" / "output" / "ExportBarcodes" / "barcodes.csv"

FORCE_RECOMPUTE = False

PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 9

print(f"BC555_DIR: {BC555_DIR}")
print(f"BC553_DIR: {BC553_DIR}")

## 3 — Helper functions

Defined once, called once per dataset (sections 4-9 for `BC555_sample_05/epi`,
10-15 for `BC553_sample_02/epi`) -- keeps the per-dataset sections below to
"call the helper, cache, display" rather than duplicating logic, following
`02_test_foci_density.ipynb`'s own section-3 convention.

In [ ]:
def resolve_dataset(sample_dir):
    """(sample_name, config, meta, round_info) for one experiment folder."""
    sample_name, imaging_dir = resolve_sample_identity(sample_dir / "MERci")
    positions_tag = positions_file_tag(sample_name, imaging_dir)
    config = ExperimentConfig(
        data_dir       = sample_dir / "data",
        metadata_dir   = sample_dir / "metadata",
        analysis_dir   = sample_dir / "analysis",
        settings_dir   = sample_dir / "settings",
        round_info_csv = sample_dir / "metadata" / "round_info.csv",
        positions_txt  = sample_dir / "positions" / f"positions_{positions_tag}.txt",
        image_suffix   = IMAGE_SUFFIX,
    )
    meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                    image_suffix=config.image_suffix)
    round_info = pd.read_csv(config.round_info_csv)
    return sample_name, config, meta, round_info


def estimate_coverage_threshold(fractions, bins=COVERAGE_HIST_BINS):
    """Valley between the two most prominent peaks of a tissue_fraction
    histogram -- same 'find the dip between two modes' idea as
    acquisition.mosaic._estimate_bimodal_threshold, adapted to linear-space
    fraction values (0-1) here rather than log10 pixel intensity, since
    tissue_fraction is already a bounded, unit-free ratio. Falls back to
    Otsu's method if the distribution isn't clearly bimodal (mirrors
    acquisition.mosaic._classify_tiles_by_signal's own Otsu fallback)."""
    counts, edges = np.histogram(fractions, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    peaks, props = find_peaks(counts, prominence=max(counts.max() * 0.05, 1))
    if len(peaks) >= 2:
        top2 = sorted(peaks[np.argsort(props["prominences"])[::-1][:2]])
        lo_idx, hi_idx = top2
        valley_idx = lo_idx + int(np.argmin(counts[lo_idx:hi_idx + 1]))
        return float(centers[valley_idx])
    return float(threshold_otsu(np.asarray(fractions)))


def plot_coverage_distribution(fractions, threshold, label, save_path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(fractions, bins=COVERAGE_HIST_BINS, color="steelblue", edgecolor="white")
    ax.axvline(threshold, color="red", linestyle="--", lw=1.5, label=f"threshold = {threshold:.3f}")
    ax.set_xlabel("Tissue fraction (per FOV)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Number of FOVs", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{label} -- tissue fraction distribution ({len(fractions)} FOVs)", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()


def compute_bit_intensity_wide(config, meta, bit_lookup, bits_series_by_round, fov_ids,
                                cache_path, label, force_recompute):
    """Per-(selected FOV, bit) mean intensity, long format cached to *cache_path*
    (same shape/columns as 04_test_intensity_vs_bit.ipynb section 5), pivoted
    to wide (fov_id x bit) and restricted to *fov_ids* on return."""
    if cache_path.exists() and not force_recompute:
        cache_df = pd.read_csv(cache_path)
    else:
        cache_df = pd.DataFrame(columns=["fov_id", "bit", "mean_intensity"])

    n_bits = len(bit_lookup)
    done_fovs = {fov_id for fov_id, n in cache_df.groupby("fov_id").size().items() if n == n_bits}
    todo_fov_ids = [f for f in fov_ids if f not in done_fovs]
    print(f"[{label}] {len(done_fovs & set(fov_ids))} / {len(fov_ids)} selected FOV(s) already "
          f"cached; computing {len(todo_fov_ids)} more.")

    new_rows = []
    reporter = ProgressReporter(total=len(todo_fov_ids), label=f"[{label}] Computing per-FOV, per-bit intensity")
    for fov_id in reporter.wrap(todo_fov_ids):
        for bit, (round_id, frame_idx) in bit_lookup.items():
            series = bits_series_by_round[round_id]
            path   = series.resolve_path(fov_id, config.image_suffix)
            frame  = read_image_frames(path, [frame_idx])[0]
            new_rows.append({"fov_id": fov_id, "bit": bit, "mean_intensity": float(frame.mean())})

    if new_rows:
        cache_df = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)
        cache_df = cache_df.drop_duplicates(subset=["fov_id", "bit"]).sort_values(["fov_id", "bit"]).reset_index(drop=True)
        cache_df.to_csv(cache_path, index=False)

    wide = (cache_df[cache_df["fov_id"].isin(fov_ids)]
            .pivot(index="fov_id", columns="bit", values="mean_intensity").sort_index())
    print(f"[{label}] Intensity available for {wide.shape[0]} / {len(fov_ids)} selected FOV(s) x {wide.shape[1]} bit(s).")
    return wide


def align_bit_profiles(wide):
    """Same least-squares affine overlay as 04_test_intensity_vs_bit.ipynb
    section 7: per-bit min-subtract, per-bit [0,1]-normalize, average across
    bits into one consensus shape, then fit each bit's own min-subtracted
    profile onto that consensus with a single no-intercept least-squares
    scale factor (Factor_bitX)."""
    min_per_bit   = wide.min(axis=0)
    range_per_bit = wide.max(axis=0) - min_per_bit
    centered      = wide.subtract(min_per_bit, axis=1)          # y - min(y), per bit
    normalized_01 = centered.divide(range_per_bit, axis=1)      # (y - min) / (max - min), per bit
    consensus     = normalized_01.mean(axis=1).values           # average shape across all bits

    factor_per_bit = {}
    for b in wide.columns:
        y = centered[b].values
        factor_per_bit[b] = float(np.dot(y, consensus) / np.dot(y, y))
    factor_series = pd.Series(factor_per_bit).sort_index()

    scaled = centered.multiply(factor_series, axis=1)   # Factor_bitX * (y - min(y))
    return scaled, factor_series


def plot_bit_profiles(fov_index, bits, values_df, color_of, ylabel, title, save_path):
    fig, ax = plt.subplots(figsize=(11, 5))
    for b in bits:
        ax.plot(fov_index, values_df[b].values, "-", lw=0.8, alpha=0.8, color=color_of[b], label=f"bit {b:02d}")
    ax.set_xlabel("FOV index (sorted by fov_id, tissue-coverage-filtered)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, ncol=2, loc="upper right")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()


def plot_aligned_with_average(fov_index, bits, scaled_df, title, save_path):
    """Redraws the aligned per-bit profiles (thin grey, alpha=0.5) with their
    per-FOV mean overlaid as one bold curve -- the weighted average intensity
    profile. Returns that average (one value per FOV, in *fov_index* order):
    this notebook's refined per-FOV WAI."""
    average_profile = scaled_df.mean(axis=1).values
    fig, ax = plt.subplots(figsize=(11, 5))
    for b in bits:
        ax.plot(fov_index, scaled_df[b].values, "-", lw=0.8, alpha=0.5, color="0.6")
    ax.plot(fov_index, average_profile, "-", lw=2.0, color="crimson", label="weighted average intensity profile")
    ax.set_xlabel("FOV index (sorted by fov_id, tissue-coverage-filtered)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Affine-scaled intensity", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    return average_profile


def plot_wai_heatmap(fov_ids, wai_values, meta, label, save_path):
    """Same layout idiom as 01_test_weighted_average_intensity.ipynb section 6:
    stage (x, y) -> integer grid index, filled with each FOV's WAI."""
    grid = positions_to_grid_indices(fov_ids, meta)
    n_x  = max(xi for xi, _ in grid.values()) + 1
    n_y  = max(yi for _, yi in grid.values()) + 1

    matrix = np.full((n_y, n_x), np.nan)
    for fov_id, wai in zip(fov_ids, wai_values):
        xi, yi = grid[fov_id]
        matrix[yi, xi] = wai

    fig, ax = plt.subplots(figsize=(max(9, n_x * 0.7 + 3), max(4, n_y * 0.4 + 1.5)))
    im = ax.imshow(matrix, cmap="viridis", origin="upper")
    ax.set_title(f"{label} -- aligned WAI per FOV ({len(fov_ids)} coverage-selected FOVs)", fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xlabel("X grid index  (increasing stage X \u2192)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Y grid index  (increasing stage Y \u2193)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = fig.colorbar(im, cax=cax, label="Aligned WAI (mean of 16 affine-scaled bit intensities)")
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()


def plot_wai_histogram(wai_values, label, save_path):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(wai_values, bins=30, color="steelblue", edgecolor="white")
    ax.set_xlabel("Aligned WAI (mean of 16 affine-scaled bit intensities)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Number of FOVs", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"{label} -- aligned WAI distribution ({len(wai_values)} FOVs)", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    print(f"[{label}] Aligned WAI: mean={np.mean(wai_values):.4f}  std={np.std(wai_values):.4f}  "
          f"cv={np.std(wai_values) / np.mean(wai_values):.4f}")

## 4 — Resolve `BC555_sample_05/epi`

In [ ]:
sample_name_555, config_555, meta_555, round_info_555 = resolve_dataset(BC555_DIR)

cache_dir_555 = config_555.analysis_dir / "cache" / NOTEBOOK_NAME
cache_dir_555.mkdir(parents=True, exist_ok=True)
figures_dir_555 = get_merci_figures_dir(BC555_DIR, "tests", NOTEBOOK_NAME, subfolder="optimization_fov_selection")
tissue_cache_dir_555 = config_555.analysis_dir / "cache" / TISSUE_CACHE_NAME

bit_lookup_555, bits_series_555, z_used_555 = resolve_barcode_bit_lookup(
    config_555, meta_555, round_info_555, n_barcode_bits=N_BARCODE_BITS, z_plane=Z_PLANE,
)

print(f"Sample name: {sample_name_555}")
print(f"FOVs       : {meta_555.n_fovs}")
print(f"Z_PLANE    : {z_used_555}")

## 5 — Step 1: tissue coverage (`BC555_sample_05/epi`)

Per-FOV `tissue_fraction` via `compute_tissue_fraction` (true-pixel-count
method, threshold = highest pixel value among the `N_BACKGROUND_FRAMES`
dimmest frames), then a coverage-selection threshold on that per-FOV
distribution itself (`estimate_coverage_threshold`, section 3).

In [ ]:
tissue_df_555, pixel_threshold_555 = compute_tissue_fraction(
    config_555, meta_555, tissue_cache_dir_555, "BC555_sample_05/epi",
    n_background_frames=N_BACKGROUND_FRAMES,
)
coverage_threshold_555 = estimate_coverage_threshold(tissue_df_555["tissue_fraction"].values)
selected_555 = sorted(tissue_df_555.loc[tissue_df_555["tissue_fraction"] >= coverage_threshold_555, "fov_id"])

print(f"[BC555_sample_05/epi] pixel threshold={pixel_threshold_555:.1f}  "
      f"coverage threshold={coverage_threshold_555:.3f} -- "
      f"{len(selected_555)} / {len(tissue_df_555)} FOV(s) selected as 'full'.")

plot_coverage_distribution(tissue_df_555["tissue_fraction"].values, coverage_threshold_555,
                            "BC555_sample_05/epi", figures_dir_555 / f"{NOTEBOOK_NAME}.coverage_distribution.png")

## 6 — Step 2: per-bit intensity, before alignment (`BC555_sample_05/epi`)

Restricted to `selected_555` (the coverage-filtered FOVs from section 5) --
same calculation as `04_test_intensity_vs_bit.ipynb` section 5, plotted the
same way as its section 6.

In [ ]:
wide_555 = compute_bit_intensity_wide(
    config_555, meta_555, bit_lookup_555, bits_series_555, selected_555,
    cache_dir_555 / "intensity_per_fov_per_bit.csv", "BC555_sample_05/epi", FORCE_RECOMPUTE,
)
fov_index_555 = np.arange(len(wide_555))
bits_555      = sorted(wide_555.columns)
color_of_555  = {b: plt.cm.turbo(i / max(len(bits_555) - 1, 1)) for i, b in enumerate(bits_555)}

plot_bit_profiles(fov_index_555, bits_555, wide_555, color_of_555, "Mean intensity",
                   f"BC555_sample_05/epi -- per-bit mean intensity vs. FOV, coverage-filtered (z={z_used_555})",
                   figures_dir_555 / f"{NOTEBOOK_NAME}.intensity_per_bit.png")

## 7 — Step 2: after alignment + weighted average intensity profile (`BC555_sample_05/epi`)

Affine overlay (section 3's `align_bit_profiles`, same math as
`04_test_intensity_vs_bit.ipynb` section 7) plotted the same way as that
notebook's own section 7, then a third plot redrawing those same aligned
profiles in grey with their per-FOV average on top -- the **weighted
average intensity profile**, and this notebook's refined per-FOV WAI.

In [ ]:
scaled_555, factor_series_555 = align_bit_profiles(wide_555)

plot_bit_profiles(fov_index_555, bits_555, scaled_555, color_of_555,
                   "Affine-scaled intensity  (Factor_bitX * (y - min(y)))",
                   "BC555_sample_05/epi -- per-bit profiles after affine overlay, coverage-filtered",
                   figures_dir_555 / f"{NOTEBOOK_NAME}.intensity_overlay.png")

wai_per_fov_555 = plot_aligned_with_average(
    fov_index_555, bits_555, scaled_555,
    "BC555_sample_05/epi -- aligned profiles + weighted average intensity profile",
    figures_dir_555 / f"{NOTEBOOK_NAME}.aligned_with_average.png",
)

## 8 — Step 3: spatial heatmap of aligned WAI (`BC555_sample_05/epi`)

In [ ]:
plot_wai_heatmap(wide_555.index.tolist(), wai_per_fov_555, meta_555, "BC555_sample_05/epi",
                  figures_dir_555 / f"{NOTEBOOK_NAME}.wai_heatmap.png")

## 9 — Step 4: aligned WAI histogram (`BC555_sample_05/epi`)

In [ ]:
plot_wai_histogram(wai_per_fov_555, "BC555_sample_05/epi",
                    figures_dir_555 / f"{NOTEBOOK_NAME}.wai_histogram.png")

## 10 — Resolve `BC553_sample_02/epi`

In [ ]:
sample_name_553, config_553, meta_553, round_info_553 = resolve_dataset(BC553_DIR)

cache_dir_553 = config_553.analysis_dir / "cache" / NOTEBOOK_NAME
cache_dir_553.mkdir(parents=True, exist_ok=True)
figures_dir_553 = get_merci_figures_dir(BC553_DIR, "tests", NOTEBOOK_NAME, subfolder="optimization_fov_selection")
tissue_cache_dir_553 = config_553.analysis_dir / "cache" / TISSUE_CACHE_NAME

bit_lookup_553, bits_series_553, z_used_553 = resolve_barcode_bit_lookup(
    config_553, meta_553, round_info_553, n_barcode_bits=N_BARCODE_BITS, z_plane=Z_PLANE,
)

print(f"Sample name: {sample_name_553}")
print(f"FOVs       : {meta_553.n_fovs}")
print(f"Z_PLANE    : {z_used_553}")

## 11 — Step 1: tissue coverage (`BC553_sample_02/epi`)

In [ ]:
tissue_df_553, pixel_threshold_553 = compute_tissue_fraction(
    config_553, meta_553, tissue_cache_dir_553, "BC553_sample_02/epi",
    n_background_frames=N_BACKGROUND_FRAMES,
)
coverage_threshold_553 = estimate_coverage_threshold(tissue_df_553["tissue_fraction"].values)
selected_553 = sorted(tissue_df_553.loc[tissue_df_553["tissue_fraction"] >= coverage_threshold_553, "fov_id"])

print(f"[BC553_sample_02/epi] pixel threshold={pixel_threshold_553:.1f}  "
      f"coverage threshold={coverage_threshold_553:.3f} -- "
      f"{len(selected_553)} / {len(tissue_df_553)} FOV(s) selected as 'full'.")

plot_coverage_distribution(tissue_df_553["tissue_fraction"].values, coverage_threshold_553,
                            "BC553_sample_02/epi", figures_dir_553 / f"{NOTEBOOK_NAME}.coverage_distribution.png")

## 12 — Step 2: per-bit intensity, before alignment (`BC553_sample_02/epi`)

In [ ]:
wide_553 = compute_bit_intensity_wide(
    config_553, meta_553, bit_lookup_553, bits_series_553, selected_553,
    cache_dir_553 / "intensity_per_fov_per_bit.csv", "BC553_sample_02/epi", FORCE_RECOMPUTE,
)
fov_index_553 = np.arange(len(wide_553))
bits_553      = sorted(wide_553.columns)
color_of_553  = {b: plt.cm.turbo(i / max(len(bits_553) - 1, 1)) for i, b in enumerate(bits_553)}

plot_bit_profiles(fov_index_553, bits_553, wide_553, color_of_553, "Mean intensity",
                   f"BC553_sample_02/epi -- per-bit mean intensity vs. FOV, coverage-filtered (z={z_used_553})",
                   figures_dir_553 / f"{NOTEBOOK_NAME}.intensity_per_bit.png")

## 13 — Step 2: after alignment + weighted average intensity profile (`BC553_sample_02/epi`)

In [ ]:
scaled_553, factor_series_553 = align_bit_profiles(wide_553)

plot_bit_profiles(fov_index_553, bits_553, scaled_553, color_of_553,
                   "Affine-scaled intensity  (Factor_bitX * (y - min(y)))",
                   "BC553_sample_02/epi -- per-bit profiles after affine overlay, coverage-filtered",
                   figures_dir_553 / f"{NOTEBOOK_NAME}.intensity_overlay.png")

wai_per_fov_553 = plot_aligned_with_average(
    fov_index_553, bits_553, scaled_553,
    "BC553_sample_02/epi -- aligned profiles + weighted average intensity profile",
    figures_dir_553 / f"{NOTEBOOK_NAME}.aligned_with_average.png",
)

## 14 — Step 3: spatial heatmap of aligned WAI (`BC553_sample_02/epi`)

In [ ]:
plot_wai_heatmap(wide_553.index.tolist(), wai_per_fov_553, meta_553, "BC553_sample_02/epi",
                  figures_dir_553 / f"{NOTEBOOK_NAME}.wai_heatmap.png")

## 15 — Step 4: aligned WAI histogram (`BC553_sample_02/epi`)

In [ ]:
plot_wai_histogram(wai_per_fov_553, "BC553_sample_02/epi",
                    figures_dir_553 / f"{NOTEBOOK_NAME}.wai_histogram.png")

## 16 — Step 5: aligned WAI vs. MERlin barcode count (`BC555_sample_05/epi` only)

`ExportBarcodes/barcodes.csv`'s `fov` column, grouped and counted, gives each
FOV's total decoded barcode count -- assumes MERlin's own `fov` numbering
matches MERci's `fov_id` (see section 0). Restricted to the same
coverage-selected, aligned-WAI FOVs as sections 5-9.

In [ ]:
barcodes_555 = pd.read_csv(MERLIN_BARCODES_PATH_555)
barcode_counts_555 = barcodes_555.groupby("fov").size().rename("n_barcodes")

wai_series_555 = pd.Series(wai_per_fov_555, index=wide_555.index, name="wai")
comparison_555 = pd.DataFrame(wai_series_555).join(barcode_counts_555, how="left")
comparison_555["n_barcodes"] = comparison_555["n_barcodes"].fillna(0)

r_555 = np.corrcoef(comparison_555["wai"], comparison_555["n_barcodes"])[0, 1]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(comparison_555["wai"], comparison_555["n_barcodes"], s=20, alpha=0.6, color="steelblue")
ax.set_xlabel("Aligned WAI (weighted average intensity profile)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("Total barcode count (MERlin ExportBarcodes)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"BC555_sample_05/epi -- aligned WAI vs. MERlin barcode count -- Pearson r={r_555:.3f}",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()
fig.savefig(figures_dir_555 / f"{NOTEBOOK_NAME}.wai_vs_barcode_count.png", dpi=150)
plt.show()

print(comparison_555.describe().to_string())
print(f"\nPearson r (aligned WAI vs. MERlin barcode count): {r_555:.4f}")